<a href="https://colab.research.google.com/github/fabianoborgesbsb/Cognitive-Proximity_OpenAlex_Gravity_Model/blob/main/Cognitive_distance__BR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cognitive and geographical distances of scholarly institutions

# Preparation

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('Authenticated')

In [ ]:
import numpy as np
from numpy import (array, dot, arccos, clip)
from numpy.linalg import norm
import pandas as pd
import pandas_gbq as pgbq
import geopy.distance

# Institutions and their works in clusters

Number of works in every cluster classifications for every institution.

In [ ]:
%%bigquery --project=multiobs

CREATE OR REPLACE TABLE projectdb_cognitive_distance_br_2026.institutions_topics AS (
    SELECT
      authorships.institution_id,
      topics.topic_id,
      COUNT(DISTINCT(works.id)) AS nw,
      SUM(1/works.authors_count) AS nc
    FROM publicdb_openalex_2026_01_rm.works_topics AS topics
    JOIN publicdb_openalex_2026_01_rm.works AS works
    ON topics.work_id = works.id
    JOIN publicdb_openalex_2026_01_rm.works_authorships_affiliations AS authorships
    ON works.id = authorships.work_id
    WHERE works.publication_year >= 2010
      AND works.publication_year <= 2024
      AND topics.score >0.95
      AND NOT works.is_xpac
      AND works.type = 'article'
    GROUP BY authorships.institution_id, topics.topic_id
);


In [ ]:
%%bigquery --project=multiobs

CREATE OR REPLACE TABLE projectdb_cognitive_distance_br_2026.institutions_br_metrics AS (
  SELECT a.institution_id,
    a.topic_id,
    a.nw AS nw,
    a.nc AS nc,
  FROM projectdb_cognitive_distance_br_2026.institutions_topics AS a
  JOIN publicdb_openalex_2026_01_rm.institutions AS b
  ON a.institution_id = b.id
  WHERE b.works_count > 0 AND (b.country_code = 'BR'
    OR LOWER(b.country) LIKE '%brazil%'
    OR LOWER(b.country) LIKE '%brasil%')
  ORDER BY a.institution_id
)

In [ ]:
%%bigquery --project=multiobs

CREATE OR REPLACE TABLE  projectdb_cognitive_distance_br_2026.institutions_br_ids AS

 SELECT DISTINCT(institution_id)
 FROM projectdb_cognitive_distance_br_2026.institutions_br_metrics

In [ ]:
%%bigquery --project=multiobs

SELECT a.institution_id, ANY_VALUE(b.display_name) AS name,
FROM projectdb_cognitive_distance_br_2026.institutions_br_metrics AS a
JOIN projectdb_cognitive_distance_br_2026.institutions_2026 AS b
ON a.institution_id = b.institution_id
GROUP BY a.institution_id

In [ ]:
%%bigquery df --project=multiobs

SELECT *
FROM projectdb_cognitive_distance_br_2026.institutions_br_metrics

In [ ]:
df['institution_id'].unique()

Institutions with largest numbers of works -- different thresholds

In [ ]:
%%bigquery  --project=multiobs

CREATE OR REPLACE TABLE projectdb_cognitive_distance_br_2026.institutions_2026 AS (
  SELECT DISTINCT a.institution_id,
    b.display_name,
    b.latitude,
    b.longitude,
    b.works_count,
  FROM projectdb_cognitive_distance_br_2026.institutions_br_ids AS a
  JOIN publicdb_openalex_2026_01_rm.institutions AS b
  ON a.institution_id = b.id
  WHERE b.works_count > 0 AND (b.country_code = 'BR'
    OR LOWER(b.country) LIKE '%brazil%'
    OR LOWER(b.country) LIKE '%brasil%')
  ORDER BY a.institution_id
)

## Pivot to get clusters as columns

Rows withs institutions and clusters are pivoted to get one institutions every row and one cluister every column.

In [ ]:
df = df.pivot(index='institution_id', columns='topic_id', values=['nw','nc']).fillna(0)
df = df.sort_values(by='institution_id')
df.head()

In [ ]:
df

# Cognitive distance

Using scalar product (cosine) as cognitive distance betweeen every two institutions. The full matrix would be symmetric, it is not needed to calculate distances from B to A if A to B is already calculated.

In [ ]:
institutions = list(df.index)
insta = []
instb = []
cd = []
for i,inst1 in enumerate(institutions):
  print(i,inst1) # print to follow progress
  for j,inst2 in enumerate(institutions):
    if inst2>=inst1: # symmetric matrix, just A to B is needed
      a = df.loc[inst1,'nw'].to_numpy()
      b = df.loc[inst2,'nw'].to_numpy()
      c = dot(a,b)/norm(a)/norm(b)
      insta += [inst1]
      instb += [inst2]
      cd += [c]

Tcd = pd.DataFrame({'inst1':insta,
                    'inst2':instb,
                    'cognitive':cd})

In [ ]:
Tcd.dtypes

Store into BigQuery.

In [ ]:
pgbq.to_gbq(Tcd,project_id = 'multiobs',
        destination_table = 'projectdb_cognitive_distance_br_2026.institutions_cognitive_distance_br',
        if_exists = 'replace')

# Geographical distance

Filter institutions by productivity threshold.

In [ ]:
%%bigquery df --project=multiobs

SELECT DISTINCT a.institution_id,
  b.latitude,
  b.longitude,
  #b.country_code
FROM projectdb_cognitive_distance_br_2026.institutions_topics AS a
JOIN projectdb_cognitive_distance_br_2026.institutions_2026 AS b
ON a.institution_id = b.institution_id
#WHERE b.works_count > 0  AND (b.country_code = 'BR'
#    OR LOWER(b.country) LIKE '%brazil%'
#    OR LOWER(b.country) LIKE '%brasil%')

In [ ]:
%%bigquery df --project=multiobs

SELECT DISTINCT institution_id,
  latitude,
  longitude,
FROM projectdb_cognitive_distance_br_2026.institutions_2026


In [ ]:
df = df.sort_values(by='institution_id')
institutions = list(df['institution_id'])
insta = []
instb = []
gd = []

for i,inst1 in enumerate(institutions):
  print(i,inst1)
  for j,inst2 in enumerate(institutions):
    if inst2>=inst1: # symmetric matrix, just A to B is needed
      a = df.loc[df['institution_id']==inst1,['latitude','longitude']].to_numpy()
      b = df.loc[df['institution_id']==inst2,['latitude','longitude']].to_numpy()
      c = geopy.distance.geodesic(a, b).km
      insta += [inst1]
      instb += [inst2]
      gd += [c]

Tgd = pd.DataFrame({'inst1':insta,'inst2':instb,
                    'geographical':gd})

In [ ]:
df.head()

Store into BigQuery.

In [ ]:
pgbq.to_gbq(Tgd, project_id = 'multiobs',
        destination_table = 'projectdb_cognitive_distance_br_2026.institutions_geographical_distance_br',
        if_exists = 'replace')

# Collaboration distance

Count collaborations for all combinations of institutions.


In [ ]:

%%bigquery df --project=multiobs

CREATE OR REPLACE TABLE projectdb_cognitive_distance_br_2026.works_collaborations_br AS (
  SELECT a.*
  FROM multiobs.publicdb_openalex_2026_01_rm.works_authorships_affiliations AS a
  JOIN projectdb_cognitive_distance_br_2026.institutions_2026 AS b
  ON a.institution_id = b.institution_id
);

CREATE OR REPLACE TABLE projectdb_cognitive_distance_br_2026.institutions_collaborations_br AS (
 SELECT authorships1.institution_id AS inst1,
   authorships2.institution_id AS inst2,
   COUNT(DISTINCT authorships1.work_id) AS works #ERRO AQUI
 FROM projectdb_cognitive_distance_br_2026.works_collaborations_br AS authorships1
 JOIN projectdb_cognitive_distance_br_2026.works_collaborations_br AS authorships2
 ON authorships1.work_id = authorships2.work_id
 WHERE authorships1.institution_id >= authorships2.institution_id # symmetric
 GROUP BY  authorships1.institution_id,
   authorships2.institution_id
 ORDER BY authorships1.institution_id, authorships2.institution_id
)


Collaboration as number of works ($n_w = w_c$, number of distintec works in collaboration) and fractional relative colaborations ($w_a$, total works from institution $a$; $w_b$, works from $b$). It is not include in the gravity model.

$$
n_c = \frac{w_c}{\sqrt{w_a w_b}}
$$

In [ ]:

%%bigquery Tid --project=multiobs

DROP TABLE IF EXISTS projectdb_cognitive_distance_br_2026.institutions_collaboration_distance_br;
CREATE TABLE projectdb_cognitive_distance_br_2026.institutions_collaboration_distance_br AS (
 SELECT a.inst1,
   a.inst2,
   works/SQRT(b.works_count*c.works_count) AS collaboration,
   works AS w_collaboration,
   b.works_count AS inst1_works_count,
   c.works_count AS inst2_works_count
 FROM projectdb_cognitive_distance_br_2026.institutions_collaborations_br AS a
 JOIN  projectdb_cognitive_distance_br_2026.institutions_2026 AS b
 ON a.inst1 = b.institution_id
 JOIN  projectdb_cognitive_distance_br_2026.institutions_2026 AS c
 ON a.inst2 = c.institution_id
 ORDER BY a.inst1, a.inst2
)



# Join tables with pairs of institutions

Collect different types of distances in to a single table.

In [ ]:
%%bigquery --project=multiobs

DROP TABLE IF EXISTS projectdb_cognitive_distance_br_2026.institutions_all_distances_br;
CREATE TABLE projectdb_cognitive_distance_br_2026.institutions_all_distances_br AS (
  SELECT a.inst1, a.inst2,
    a.cognitive,
    b.geographical,
    c.collaboration,
    c.w_collaboration,
    c.inst1_works_count,
    c.inst2_works_count,
  FROM (
    SELECT *,
      IF(inst1<=inst2,
        CONCAT(CAST(inst1 AS STRING),'-',CAST(inst2 AS STRING)),
        CONCAT(CAST(inst2 AS STRING),'-',CAST(inst1 AS STRING))
      ) AS index_pair
    FROM projectdb_cognitive_distance_br_2026.institutions_cognitive_distance_br
  ) AS a
  LEFT JOIN (
    SELECT *,
      IF(inst1<=inst2,
        CONCAT(CAST(inst1 AS STRING),'-',CAST(inst2 AS STRING)),
        CONCAT(CAST(inst2 AS STRING),'-',CAST(inst1 AS STRING))
      ) AS index_pair
    FROM projectdb_cognitive_distance_br_2026.institutions_geographical_distance_br
  ) AS b
  ON a.index_pair = b.index_pair
  LEFT JOIN (
    SELECT *,
      IF(inst1<=inst2,
        CONCAT(CAST(inst1 AS STRING),'-',CAST(inst2 AS STRING)),
        CONCAT(CAST(inst2 AS STRING),'-',CAST(inst1 AS STRING))
      ) AS index_pair
    FROM projectdb_cognitive_distance_br_2026.institutions_collaboration_distance_br
  ) AS c
  ON a.index_pair = c.index_pair
)


In [ ]:
%%bigquery df --project=multiobs


SELECT
  a.*,
  b.display_name AS inst1_name,
  b.city AS inst1_city,
  c.display_name AS inst2_name,
  c.city AS inst2_city
FROM projectdb_cognitive_distance_br_2026.institutions_all_distances_br AS a
JOIN publicdb_openalex_2026_01_rm.institutions AS b
  ON a.inst1 = b.id
JOIN publicdb_openalex_2026_01_rm.institutions AS c
  ON a.inst2 = c.id

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
from google.colab import files
df.to_csv('cognitive_2026.csv', index=False)
files.download('cognitive_2026.csv')

Test

The collaboration matrix covers both inter- and intra-institutional interactions to reflect the complete scientific landscape. The set of potential pairs is defined by $\frac{n(n+1)}{2}$, enabling the inclusion of self-loops as proxies for internal institutional production

In [ ]:
%%bigquery --project=multiobs

-- Total original
SELECT COUNT(*) AS total_original
FROM projectdb_cognitive_distance_br_2026.institutions_all_distances_br;




In [ ]:
%%bigquery --project=multiobs

SELECT COUNT(*) AS total_join
FROM projectdb_cognitive_distance_br_2026.institutions_all_distances_br AS a
JOIN publicdb_openalex_2026_01_rm.institutions AS b
  ON a.inst1 = b.id
JOIN publicdb_openalex_2026_01_rm.institutions AS c
  ON a.inst2 = c.id;

In [ ]:
%%bigquery --project=multiobs

SELECT COUNT(*) AS lost_pairs
FROM projectdb_cognitive_distance_br_2026.institutions_all_distances_br AS a
LEFT JOIN publicdb_openalex_2026_01_rm.institutions AS b
  ON a.inst1 = b.id
LEFT JOIN publicdb_openalex_2026_01_rm.institutions AS c
  ON a.inst2 = c.id
WHERE b.id IS NULL OR c.id IS NULL;

# Cross-Country collaboration

1	Switzerland
2	Netherlands
3	Sweden
4	UK
5	Germany
6	Italy
7	France
8	Spain
9	Belgium
10	Russia
11	Poland
12	Brazil
13	Australia
14	South Korea
15	Japan
16	China
17	India
18	Taiwan
19	Canada
20	US
21	Israel
22	Iran
23	Turkey

In [ ]:
%%bigquery --project=multiobs

CREATE OR REPLACE TABLE projectdb_cognitive_distance_br_2026.institutions_countries AS
SELECT *
FROM publicdb_openalex_2026_01_rm.institutions
WHERE country_code IN ('CH','NL','SE','GB','DE','IT','FR','ES','BE','RU','PL','BR',
  'AU','KR','JP','CN','IN','TW','CA','US','IL','IR','TR')




Authorships

In [ ]:
%%bigquery --project=multiobs

CREATE OR REPLACE TABLE projectdb_cognitive_distance_br_2026.authorships AS
SELECT DISTINCT
  a.work_id,
  a.institution_id,
  b.country_code,
  w.publication_year
FROM multiobs.publicdb_openalex_2026_01_rm.works_authorships_affiliations a
JOIN projectdb_cognitive_distance_br_2026.institutions_countries b
  ON a.institution_id = b.id
JOIN multiobs.publicdb_openalex_2026_01_rm.works w
  ON a.work_id = w.id
WHERE w.publication_year BETWEEN 2010 AND 2024;

International Collaboration

This approach accounts for inter-institutional collaborations between different countries while excluding intra-institutional interactions

In [ ]:
%%bigquery --project=multiobs

CREATE OR REPLACE TABLE projectdb_cognitive_distance_br_2026.countries_collaboration AS
SELECT a.country_code AS country_code_1,
 b.country_code AS country_code_2,
 COUNT(DISTINCT a.work_id) AS Colabs_Total
FROM projectdb_cognitive_distance_br_2026.authorships AS a
JOIN  projectdb_cognitive_distance_br_2026.authorships AS b
ON a.work_id = b.work_id
WHERE a.country_code > b.country_code
GROUP BY a.country_code, b.country_code


Edges - export

In [ ]:
%%bigquery cc --project=multiobs

SELECT *
FROM projectdb_cognitive_distance_br_2026.countries_collaboration



In [ ]:
from google.colab import files
cc.to_csv('edges_STI_2026.csv', index=False)
files.download('edges_STI_2026.csv')

Nodes (based on publications)

In [ ]:
%%bigquery --project=multiobs

CREATE OR REPLACE TABLE projectdb_cognitive_distance_br_2026.countries_production AS

SELECT
  country_code AS id,
  COUNT(DISTINCT work_id) AS publications
FROM projectdb_cognitive_distance_br_2026.authorships
WHERE publication_year BETWEEN 2010 AND 2024
GROUP BY country_code;

Nodes - export

In [ ]:
%%bigquery nodes --project=multiobs

SELECT *
FROM projectdb_cognitive_distance_br_2026.countries_production



In [ ]:
from google.colab import files
nodes.to_csv('nodes_STI_2026.csv', index=False)
files.download('nodes_STI_2026.csv')
